# Fine-Tuning para Regresion: Prediciendo el Puntaje de un Vino

Todo este modulo entrenamos BERT para **clasificacion**: predecir una categoria (Positivo/Negativo) a partir de un tweet. Pero la receta de transfer learning que usamos -- congelar la base de BERT, agregar una cabeza nueva chica, entrenar solo esa cabeza -- no esta atada a la clasificacion.  
Sirve igual si en vez de una categoria queremos predecir un **numero continuo**. Eso es una **regresion**.

Para explorar esto, cambiamos de dataset: en vez de tweets con sentimiento, vamos a trabajar con reseñas de vinos escritas por sommeliers, y vamos a tratar de predecir el puntaje (entre 80 y 100) que le puso el critico, a partir unicamente del texto de la reseña.

### El plan de hoy

1. Cargar y explorar el dataset de reseñas de vino.
2. Preparar los datos: esta vez la etiqueta es un numero, no una categoria.
3. Cargar BERT configurado para regresion (`num_labels=1`), y congelar sus capas base, exactamente como hicimos con clasificacion.
4. Entrenar y evaluar, esta vez con metricas de error en vez de accuracy.
5. Probar el modelo con una reseña nueva.
6. Cerrar comparando, punto por punto, en que se parece y en que se diferencia esto de lo que hicimos con clasificacion.

## 0. Preparando el entorno

In [1]:
# Quitamos torchvision: no la necesitamos (solo trabajamos con texto), y en algunos entornos
# de Colab su version rompe la deteccion automatica que hace la libreria datasets
!pip uninstall -y -q torchvision
!pip install -q transformers datasets accelerate scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

## 1. Cargando y explorando el dataset

Este dataset viene de WineEnthusiast: cada fila es la reseña de un vino, escrita por un catador profesional, junto con el puntaje que le asigno (en una escala de 80 a 100 -- la publicacion solo reseña vinos que superan los 80 puntos).

In [3]:
url_dataset = "https://raw.githubusercontent.com/freesinger/wine_review/master/data/winemag-data_first150k.csv"

columnas_utiles = ["description", "points"]
df = pd.read_csv(url_dataset, usecols=columnas_utiles)

print("Filas:", len(df))
df.head()

Filas: 150930


,description,points
0,This tremendous 100% varietal wine hails from ...,96
1,"Ripe aromas of fig, blackberry and cassis are ...",96
2,Mac Watson honors the memory of a wine once ma...,96
3,"This spent 20 months in 30% new French oak, an...",96
4,"This is the top wine from La Bégude, named aft...",95


In [4]:
df["points"].describe()

,points
count,150930.000000
mean,87.888418
std,3.222392
min,80.000000
25%,86.000000
50%,88.000000
75%,90.000000
max,100.000000


La etiqueta puede tomar cualquier valor entero entre 80 y 100: son 21 valores posibles.  
<br>
La puntuación original está en una escala de 1 a 100, pero Wine Enthusiast no publica reseñas de vinos con menos de 80 puntos.

## 2. Preparando los datos

Nos quedamos con una muestra del dataset (tiene mas de 150,000 filas, mucho mas de lo que necesitamos para este ejercicio), separamos en train y test, y renombramos la columna de puntaje a `labels`, como espera el `Trainer`.

Un detalle importante: `labels` tiene que ser de tipo `float`, no `int`. La funcion de perdida que vamos a usar para regresion (MSE) espera numeros de punto flotante; si dejamos la columna como entero, vamos a tener problemas mas adelante.

In [6]:
tamaño_muestra = 3000 # Seleccionamos una muestra de 3000 reseñas

df_muestra = df.sample(n=tamaño_muestra, random_state=42).reset_index(drop=True)

# Convertimos la puntuación a número decimal para usarla como valor que el modelo deberá predecir
df_muestra["labels"] = df_muestra["points"].astype("float32")

# Dividimos los datos en un conjunto de entrenamiento (80%) y uno de prueba (20%)
train_df, test_df = train_test_split(
    df_muestra,
    test_size=0.2,
    random_state=42
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 2400
Test: 600


In [7]:
# Mostramos el texto de la primera reseña de la muestra
print("Reseña:")
print(df_muestra["description"].iloc[0])

print("\nPuntuación:")
print(df_muestra["points"].iloc[0])

Reseña:
A cool cherry aroma and a mild, creamy paste note open the nose. The feel is fiery and untamed, while full flavors of cherry and plum seem raw. A cocoa flavor and juicy feel define the finish.

Puntuación:
84


In [8]:
# Normalizamos la etiqueta: media 0, desvio estandar 1
media_puntaje = train_df["points"].mean()
desvio_puntaje = train_df["points"].std()

train_df["labels"] = ((train_df["points"] - media_puntaje) / desvio_puntaje).astype("float32")
test_df["labels"] = ((test_df["points"] - media_puntaje) / desvio_puntaje).astype("float32")

print("\nPuntuación normalizada:")
print(train_df["labels"].iloc[0])


Puntuación normalizada:
-1.5045379


## 3. Tokenizando el dataset

Mismo proceso que con los tweets: convertimos a `Dataset`, tokenizamos en batch, y dejamos todo en formato PyTorch. Las reseñas de vino son bastante mas largas que los tweets (en promedio, unos 240 caracteres), asi que le damos un `max_length` un poco mas generoso.

In [9]:
# Cargamos el tokenizador de BERT para convertir las reseñas de vino en tokens
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Función que tokeniza las reseñas y prepara las secuencias para BERT
def tokenizar(batch):
    return tokenizer(
        batch["description"],       # Textos que queremos convertir en tokens
        truncation=True,             # Recortamos las reseñas que superen el máximo
        padding="max_length",        # Rellenamos las reseñas cortas hasta el mismo tamaño
        max_length=96,               # Cada reseña tendrá como máximo 96 tokens
    )


# Convertimos los DataFrames en datasets compatibles con Hugging Face
train_dataset = Dataset.from_pandas(train_df[["description", "labels"]].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[["description", "labels"]].reset_index(drop=True))

# Aplicamos la tokenización a todos los ejemplos del dataset
train_dataset = train_dataset.map(tokenizar, batched=True)
test_dataset = test_dataset.map(tokenizar, batched=True)

# Indicamos las columnas que el modelo necesita para hacer sus cálculos
columnas_modelo = ["input_ids", "attention_mask", "labels"]

# Convertimos esas columnas a tensores de PyTorch
train_dataset.set_format("torch", columns=columnas_modelo)
test_dataset.set_format("torch", columns=columnas_modelo)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [10]:
# Comprobamos la puntuación (label) del primer ejemplo
train_dataset[0]["labels"]

tensor(-1.5045)

Fijate que `labels` aparece como un numero con decimales (por ejemplo `tensor(91.)`), no como un indice de categoria. Esa es la señal de que quedo bien preparado para regresion.

In [11]:
# Veamos la primera review tokenized:

# los IDs
print(train_dataset[0]["input_ids"])

# los tokens
tokens = tokenizer.convert_ids_to_tokens(train_dataset[0]["input_ids"])
print(tokens)

tensor([   101,  11919,  10107,  10169,  11419,  45079,  19899,  10108,  12361,
         11478,  10111,  17995,  10201, 103631,    117,  61010,  10741,  10155,
           187,  87192,  10108,  15176,  68462,  10238,    119,  10117,  36796,
         96182,  10216,  10124,  15765,  10111,  10798,  31214,  19706,  16988,
           117,    191,  53750,  15398,  10741,  10169,    169,  13716,  11499,
         21328,  10108,  10141,  16278,  11161,  10570,    119,    102,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0])
['[CLS]', 'Open', '##s', 'with', 'ex', '##otic', 'notes', 'of', 'mu', '##sk', 'and', 'summer', 'ja', '##smine', ',', 'backed', 'up', 'by', 's',

## 4. Cargando el modelo para regresion

Esta es la unica diferencia real al cargar el modelo: usamos `num_labels=1` en vez de 2 o 3, y agregamos `problem_type="regression"`. Con `num_labels=1`, la cabeza de clasificacion que se agrega arriba de BERT termina teniendo una sola neurona de salida -- un numero, no un vector de probabilidades por categoria. `problem_type="regression"` le confirma al modelo que la funcion de perdida a usar es MSE (error cuadratico medio) y no cross-entropy.

In [12]:
# Cargamos BERT con una nueva capa de salida preparada para regresión. A diferencia de clasificación, el modelo debe predecir un valor numérico (la puntuación del vino).
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,                  # Una única salida: la puntuación predicha
    problem_type="regression",     # Indicamos que queremos predecir un valor continuo
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


> *AutoModelForSequenceClassification* se utiliza tanto para clasificación como para regresión !  
(no existe  " *AutoModelForSequenceRegression* ")
```
AutoModelForSequenceClassification
            │
      ┌─────┴─────┐
      ▼           ▼
Classification  Regression
  num_labels=2   num_labels=1
      │              │
 Negativo/Positivo   92.5

## 5. Congelando las capas base

Exactamente el mismo paso que hicimos con clasificacion. `model.bert.parameters()` no sabe ni le importa si la cabeza de arriba predice una categoria o un numero: el congelamiento funciona igual en los dos casos.

In [13]:
# Congelamos todos los parámetros de BERT para mantener sus pesos pre-entrenados. Solo la nueva capa de regresión quedará disponible para el entrenamiento.
for param in model.bert.parameters():
    param.requires_grad = False

In [14]:
# Contamos todos los parámetros del modelo y cuántos se pueden entrenar
total = sum(p.numel() for p in model.parameters())
entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parametros totales: {total:,}")
print(f"Parametros entrenables: {entrenables:,}")
print(f"Porcentaje entrenable: {100 * entrenables / total:.5f}%")

Parametros totales: 177,854,209
Parametros entrenables: 769
Porcentaje entrenable: 0.00043%


## 6. Definiendo metricas para regresion

Medimos que tan lejos esta la prediccion del valor real:

- **MSE** (error cuadratico medio): el promedio de las diferencias al cuadrado. Penaliza mas los errores grandes.
- **RMSE** (raiz del MSE): lo mismo, pero en las unidades originales (puntos), mas facil de interpretar.
- **MAE** (error absoluto medio): el promedio de las diferencias, sin elevar al cuadrado. Mas facil de leer como "en promedio, nos equivocamos por esta cantidad de puntos".

In [15]:
# Calculamos métricas para evaluar qué tan cerca están las puntuaciones predichas por el modelo de las puntuaciones reales de los vinos.
def calcular_metricas(eval_pred):

    # Extraemos las predicciones del modelo y las puntuaciones reales.
    predicciones, etiquetas = eval_pred

    # Des-normalizamos antes de medir el error, para que MSE/RMSE/MAE queden en "puntos", no en unidades normalizadas
    predicciones = predicciones.squeeze(-1) * desvio_puntaje + media_puntaje
    etiquetas = etiquetas * desvio_puntaje + media_puntaje

    # MSE: penaliza especialmente los errores grandes
    mse = mean_squared_error(etiquetas, predicciones)

    # RMSE: Raíz del MSE
    rmse = mse ** 0.5

    # MAE: indica el error absoluto promedio en puntos
    mae = mean_absolute_error(etiquetas, predicciones)

    return {
        "mse": mse,             # Error cuadrático medio
        "rmse": rmse,           # Raíz del MSE, en la misma escala que la target variable
        "mae": mae,             # Error absoluto medio
    }

- MAE : error promedio fácil de entender
- MSE : castiga errores grandes
- RMSE : error en la misma escala que points

## 7. Configurando el entrenamiento

`TrainingArguments` y `Trainer` se usan exactamente igual que con clasificacion. Lo unico que cambio es el modelo (con su cabeza de una sola neurona) y la funcion de metricas.

In [ ]:
# Configuramos los parámetros que controlarán el entrenamiento del modelo.
argumentos_entrenamiento = TrainingArguments(

    output_dir="./resultados_bert_regresion", # Carpeta donde se guardarán los resultados
    num_train_epochs=20, # Número de veces que el modelo recorrerá todo el conjunto de entrenamiento.
    per_device_train_batch_size=16, # Batch size entrenamiento.
    per_device_eval_batch_size=16, # Batch size evaluación.
    learning_rate=1e-3,   # el default (5e-5) es demasiado lento para este problema
    eval_strategy="epoch", # Evaluamos el modelo al final de cada época.
    logging_strategy="epoch", # Mostramos información del entrenamiento al final de cada época.
    report_to="none", # No enviamos los resultados a servicios externos de seguimiento.

)

# Creamos el Trainer, que se encarga de organizar y ejecutar el entrenamiento.
trainer = Trainer(
    model=model, # Modelo BERT que vamos a entrenar.
    args=argumentos_entrenamiento, # Parámetros de entrenamiento definidos anteriormente.
    train_dataset=train_dataset, # Dataset utilizado para entrenar el modelo.
    eval_dataset=test_dataset, # Dataset utilizado para evaluar el modelo.
    compute_metrics=calcular_metricas, # Función que calcula MSE, RMSE y MAE durante la evaluación.

)

# Comenzamos el entrenamiento del modelo.
trainer.train()

Epoch,Training Loss,Validation Loss,Mse,Rmse,Mae
1,0.982256,0.787978,8.407781,2.899617,2.341517
2,0.884687,0.750383,8.006636,2.829600,2.243172
3,0.852013,0.905872,9.665716,3.108973,2.432502
4,0.823069,0.692687,7.391015,2.718642,2.171592
5,0.801849,0.679414,7.249393,2.692470,2.149903
6,0.790239,0.696956,7.436567,2.727007,2.136688
7,0.793217,0.674723,7.199334,2.683157,2.110560
8,0.777932,0.656974,7.009951,2.647631,2.105865
9,0.773809,0.683020,7.287865,2.699605,2.105004
10,0.774142,0.651139,6.947693,2.635848,2.079721


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.7884324798583985, metrics={'train_runtime': 391.3552, 'train_samples_per_second': 122.651, 'train_steps_per_second': 7.666, 'total_flos': 2367978236928000.0, 'train_loss': 0.7884324798583985, 'epoch': 20.0})

## 8. Evaluando el modelo

In [ ]:
resultados = trainer.evaluate()
resultados

Training Loss,Validation Loss,Epoch,Mse,Rmse,Mae
0.745084,0.638582,20,6.813712,2.610309,2.051704


{'eval_loss': 0.6385820508003235,
 'eval_mse': 6.8137124914074265,
 'eval_rmse': 2.6103088881217538,
 'eval_mae': 2.051704107755795}

Interpretemos el `rmse`: si por ejemplo da 3.0, quiere decir que en promedio nuestras predicciones se alejan unos 3 puntos del puntaje real, en una escala de 80 a 100. No esta mal para una cabeza que solo tiene una neurona entrenada, sobre una muestra chica de datos.

## 9. Inferencia con una reseña nueva

Con clasificacion, al final aplicabamos softmax y nos quedabamos con la categoria de mayor probabilidad. Con regresion no hay softmax ni argmax: la salida del modelo (el logit) *es* directamente la prediccion. Un solo numero.

In [ ]:
# Creamos una nueva reseña de vino que queremos evaluar con el modelo.
reseña_nueva = (
    "A rich and elegant wine, with dark cherry, cedar and a hint of "
    "chocolate on the nose. Full-bodied with silky tannins and a long, "
    "polished finish."
)

reseña_nueva = (
    "This wine was just horrible, very bad taste"
)


# Ponemos el modelo en modo evaluación.
model.eval()
# Convertimos la reseña en tokens y tensores que BERT pueda procesar.
encoding = tokenizer(reseña_nueva, return_tensors="pt", truncation=True, padding=True)

# Movemos los datos de entrada al mismo dispositivo que el modelo (GPU o CPU).
encoding = {k: v.to(model.device) for k, v in encoding.items()}

# Desactivamos el cálculo de gradientes
with torch.no_grad():
    salida = model(**encoding) # Pasamos la reseña tokenizada por el modelo para obtener su predicción.

# Extraemos el único valor predicho por el modelo. .item() convierte el tensor en un número de Python. (and un-normalize the raw output before printing)
puntaje_predicho = salida.logits.item() * desvio_puntaje + media_puntaje

print(f"Reseña: {reseña_nueva}")
print(f"Puntaje predicho: {puntaje_predicho:.1f}")

Reseña: A rich and elegant wine, with dark cherry, cedar and a hint of chocolate on the nose. Full-bodied with silky tannins and a long, polished finish.
Puntaje predicho: 87.4


Proba con tus propias reseñas -- alguna muy elogiosa, alguna mas tibia -- y fijate si el puntaje predicho tiene sentido. Tambien fijate que nada le impide al modelo predecir un numero fuera del rango 80-100 (por ejemplo, 102 o 76): a diferencia de la clasificacion, donde softmax garantiza una distribucion valida entre categorias, la regresion no tiene ninguna restriccion asi incorporada. Si eso te importa para un caso de uso real, es algo que se puede corregir despues (por ejemplo, recortando el valor al rango valido antes de mostrarlo).

## 10. Clasificacion vs. regresion, lado a lado

| | Clasificacion (lo que hicimos con los tweets) | Regresion (lo que acabamos de hacer) |
|---|---|---|
| `num_labels` | 2 o 3 (una por categoria) | 1 |
| `problem_type` | por defecto (`single_label_classification`) | `"regression"` |
| Tipo de `labels` | enteros (indices de categoria) | flotantes |
| Funcion de perdida | cross-entropy | MSE |
| Salida del modelo | logits, uno por categoria | un unico numero |
| Post-procesamiento | softmax + argmax | ninguno: el logit ya es la prediccion |
| Metrica principal | accuracy | MSE / RMSE / MAE |
| Congelamiento de capas base | igual | igual |
| `Trainer` / `TrainingArguments` | igual | igual |

La fila mas importante de esta tabla es la ultima: toda la infraestructura de fine-tuning -- congelar capas, `Trainer`, `TrainingArguments`, el ciclo de entrenamiento -- es identica. Lo unico que cambia es la forma de la cabeza final y como se mide el error. Esa es la idea central del transfer learning: la parte cara y dificil de entrenar (entender lenguaje) se resuelve una sola vez, y despues se reutiliza para cualquier tarea que se le ocurra poner encima, sea clasificar sentimiento o predecir un puntaje.

## Resumen

- Regresion y clasificacion comparten exactamente la misma receta de transfer learning: congelar `model.bert` y entrenar solo una cabeza nueva chica.
- Lo que cambia es la cabeza (`num_labels=1`, `problem_type="regression"`), el tipo de dato de las etiquetas (float, no int), la funcion de perdida (MSE en vez de cross-entropy), y como se interpreta la salida (un numero directo, sin softmax).
- Usamos un dataset nuevo -- reseñas de vino -- para practicar esto sobre un problema de regresion genuino: predecir un puntaje de 80 a 100 a partir del texto de la reseña.
- Medimos el desempeño con MSE, RMSE y MAE en vez de accuracy, y probamos el modelo con una reseña nueva.

Con este notebook, el modulo cubrio las dos grandes familias de tareas supervisadas de NLP con BERT: clasificar texto en categorias, y predecir un valor numerico a partir de texto. La misma base (BERT congelado) sirve para las dos.